In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from pypdf import PdfReader

sys.path.insert(0, "..")
from schema import PurchaseOrder

load_dotenv(find_dotenv(usecwd=True))
llm = OpenAI(base_url="http://localhost:4000", api_key=os.environ["LITELLM_MASTER_KEY"])
DATA = Path("../data")

In [2]:
def pdf_text(path: Path) -> str:
    return "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)


text = pdf_text(DATA / "pos" / "PO-0001.pdf")
print(text)

Salzach Kunststofftechnik GmbH · Münchner Bundesstraße 140, 5020 Salzburg, Austria
Bestellung
Alpentech Industriebedarf GmbH
Lagerstraße 5
4600 Wels
Austria
Bestellnummer:
B-26/0852
Bestelldatum:
21.01.2026
Kunden-Nr. bei Ihnen:
1000107
Liefertermin:
10.02.2026
Lieferanschrift:
Salzach Kunststofftechnik GmbH
Wareneingang
Münchner Bundesstraße 140
5020 Salzburg
Austria
Sehr geehrte Damen und Herren, wir bestellen zu unseren Allgemeinen Einkaufsbedingungen:
Pos.
Artikelnr.
Bezeichnung
Menge
ME
Einzelpreis
Gesamtpreis
1
Keilriemen SPZ 1250
13
PC
9,80
127,40
2
400120
Stretchfolie 500 mm x 300 m, 23 my
45
ROL
16,40
738,00
3
400101
Sechskantschraube DIN 933 M8x30 verzinkt,
100 Stk
15
PAK
14,90
223,50
4
Rollenkette 08B-1 Simplex, 5 m
18
PC
54,00
972,00
Gesamtsumme netto (EUR):
2.060,90
Bitte geben Sie unsere Bestellnummer auf allen Lieferpapieren und Rechnungen an.
Mit freundlichen Grüßen
Einkauf



In [3]:
SYSTEM_PROMPT = f"""You extract purchase orders into JSON.
Rules:
- Return only JSON matching this JSON schema: {json.dumps(PurchaseOrder.model_json_schema())}
- Dates as YYYY-MM-DD.
- Numbers as plain decimals with a dot and no thousands separator, e.g. 2060.90.
- Copy text fields exactly as printed. Do not translate.
- If a field is not on the document, use null.
- line_no starts at 1, in document order.
"""


def extract(text: str, doc_id: str) -> PurchaseOrder:
    response = llm.chat.completions.create(
        model="o2c-extract",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        extra_body={"metadata": {"trace_name": "extract", "tags": ["poc", doc_id]}},
    )
    return PurchaseOrder.model_validate_json(response.choices[0].message.content)

In [4]:
def compare(doc_id: str) -> None:
    result = extract(pdf_text(DATA / "pos" / f"{doc_id}.pdf"), doc_id)
    truth = PurchaseOrder.model_validate_json(
        (DATA / "ground_truth" / f"{doc_id}.json").read_text(encoding="utf-8")
    )
    print(doc_id, "MATCH" if result == truth else "DIFFERENT")
    for field in PurchaseOrder.model_fields:
        if getattr(result, field) != getattr(truth, field):
            print(
                f"  {field}:\n    got:      {getattr(result, field)}\n    expected: {getattr(truth, field)}"
            )


compare("PO-0001")

PO-0001 DIFFERENT
  customer_name:
    got:      Alpentech Industriebedarf GmbH
    expected: Salzach Kunststofftechnik GmbH
  ship_to_address:
    got:      Salzach Kunststofftechnik GmbH
Wareneingang
Münchner Bundesstraße 140
5020 Salzburg
Austria
    expected: Münchner Bundesstraße 140, 5020 Salzburg, Austria


In [5]:
for doc_id in ["PO-0002", "PO-0003", "PO-0004", "PO-0005", "PO-0006"]:
    compare(doc_id)

PO-0002 DIFFERENT
  customer_name:
    got:      Alpentech Industriebedarf GmbH
    expected: Salzach Kunststofftechnik GmbH
  ship_to_address:
    got:      Salzach Kunststofftechnik GmbH
Wareneingang
Münchner Bundesstraße 140
5020 Salzburg
Austria
    expected: Münchner Bundesstraße 140, 5020 Salzburg, Austria
  lines:
    got:      [LineItem(line_no=1, material_number=None, description='Rillenkugellager 6205-2RS C3', quantity=Decimal('17'), unit='PC', unit_price=Decimal('8.75'), line_total=Decimal('148.75')), LineItem(line_no=2, material_number='400103', description='Unterlegscheibe DIN 125 A 8,4 verzinkt, 100', quantity=Decimal('47'), unit='PAK', unit_price=Decimal('3.2'), line_total=Decimal('150.4')), LineItem(line_no=3, material_number=None, description='Hydrauliköl HLP 46', quantity=Decimal('20'), unit='L', unit_price=Decimal('3.45'), line_total=Decimal('69')), LineItem(line_no=4, material_number='400125', description='LED-Hallenstrahler 150 W IP65', quantity=Decimal('25'), unit